In [1]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from numba import jit

from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import os
from scipy.signal import find_peaks
from scipy.integrate import quad
from scipy.optimize import root_scalar
from matplotlib.animation import PillowWriter, FuncAnimation

In [2]:
# Independent parameters (free to edit)

Na = 0.5 # Units: M 
T = 303.15 # Units: K
valence = 4
duration = 500 * 10**5 # In timesteps of dt
gridpoints = 128 # Number of points
dx = 10 # Units: nm
dt = 1.E-5 # Units: sec
rho_mean_multicomponent = 6E-5 # Initial mean density of nanostar A, found by spinodal (rho dense + rho dilute)/2 for the value of T used
Cx = 1 #Ratio of AABB nanostars to AAAA and BBBB
save_interval = 10**5
inv_dx2= 1.0 / (dx * dx)


#Establishes constants
K = 1.0E6 # Units: nm^5 
M = 1 # Units: (nm s)^-1

# check effect
B2 = 2190 # Units: nm^3
vb = 1.66 # Units: nm^3


kB = 1.314E-23*0.24 # Units: cal/K (1J=0.24cal)
mol = 6.02E23
dHa = -42000 # Units: cal/mol 
dS1 = 1.84*np.log(Na) # Units: cal/mol K
dS0 = -120 # Units: cal/mol K at 1M NaCl
floor = 1E-12 # Minimum value for arrays
num_saves = duration // save_interval + 1 #Number of saved values

Da = vb * np.exp(-(dHa - T * (dS0 + dS1)) / (mol * kB * T))
Db = Da

In [3]:
@jit(nopython=True, cache = False) # Converts the given function into machine code (optimization)
def compute_step_three(rhoAAAA, rhoBBBB, rhoAABB):
    
    # Eliminates zeros/negatives
    rhoAAAA_floored = np.maximum(rhoAAAA, floor)
    rhoBBBB_floored = np.maximum(rhoBBBB, floor)
    rhoAABB_floored = np.maximum(rhoAABB, floor)

    # Initializes Xa
    CaDa = 4*(rhoAAAA_floored+0.5*rhoAABB_floored)*Da
    Xa = (-1 + np.sqrt(1 + 4 * CaDa)) / (2 * CaDa)

    #Initializes Xb
    CbDb = 4*(rhoBBBB_floored+0.5*rhoAABB_floored)*Db
    Xb = (-1 + np.sqrt(1 + 4 * CbDb)) / (2 * CbDb)

    # Calculates necessary Laplacians
    laplacian_1d_rhoAAAA = (np.roll(rhoAAAA_floored, -1) - 2.0 * rhoAAAA_floored + np.roll(rhoAAAA_floored, 1)) * inv_dx2
    laplacian_1d_rhoBBBB = (np.roll(rhoBBBB_floored, -1) - 2.0 * rhoBBBB_floored + np.roll(rhoBBBB_floored, 1)) * inv_dx2
    laplacian_1d_rhoAABB = (np.roll(rhoAABB_floored, -1) - 2.0 * rhoAABB_floored + np.roll(rhoAABB_floored, 1)) * inv_dx2

    rho_total = rhoAAAA_floored + rhoBBBB_floored + rhoAABB_floored

    # Total chemical potential of nanostar AAAA (with floored rho, Xa)
    beta_mu_total_AAAA = (2.0 * B2 * rho_total + np.log(rhoAAAA_floored) + #beta mu_ref *** USE RHO TOTAL NOT RHO_i
                        valence * np.log(Xa) - #beta mu_b
                        K * laplacian_1d_rhoAAAA) #beta mu_int
    
    # Total chemical potential of nanostar BBBB (with floored rho, Xa)
    beta_mu_total_BBBB = (2.0 * B2 * rho_total + np.log(rhoBBBB_floored) + #beta mu_ref
                        valence * np.log(Xb) - #beta mu_b
                        K * laplacian_1d_rhoBBBB) #beta mu_int
    
    # Total chemical potential of nanostar AABB (with floored rho, Xa)
    beta_mu_total_AABB = (2.0 * B2 * rho_total + np.log(rhoAABB_floored) + #beta mu_ref
                        (valence/2) * (np.log(Xa) + np.log(Xb)) - #beta mu_b
                        K * laplacian_1d_rhoAABB) #beta mu_int

    # Updates the density for each nanostar explicitly: rho(t+dt) = rho(t) + dt * M laplacian (beta mu_total)
    rhoAAAA_step = dt * M * (np.roll(beta_mu_total_AAAA, -1) - 2.0 * beta_mu_total_AAAA + np.roll(beta_mu_total_AAAA, 1)) * inv_dx2
    rhoBBBB_step = dt * M * (np.roll(beta_mu_total_BBBB, -1) - 2.0 * beta_mu_total_BBBB + np.roll(beta_mu_total_BBBB, 1)) * inv_dx2
    rhoAABB_step = dt * M * (np.roll(beta_mu_total_AABB, -1) - 2.0 * beta_mu_total_AABB + np.roll(beta_mu_total_AABB, 1)) * inv_dx2

    return(rhoAAAA_step, rhoBBBB_step, rhoAABB_step, Xa, Xb)




def calc_free_energy_multicomponent_AB(rho_total_array_AAAA, rho_total_array_BBBB, rho_total_array_AABB, Xa, Xb, v0=1):
    '''
    Finds free energy of a multi-component system using F = integral_V [ f({rho_i}) + K/2 sum_i (gradient rho)^2 ] dV

    Using f_ref = rho_tot * log(v_0 * rho_tot) - rho_tot + B2 * rho_tot^2 + sum_i [rho_i * log(rho_i/rho_tot)]

    Note that the final term IS included in this!

    Using f_b = rho_A * valence * (log(Xa) + (1-Xa)/2) + rho_B * valence * (log(Xb) + (1-Xb)/2) 
              + rho_AB * valence/2 * (log(Xa) + (1-Xa)/2) + rho_AB * valence/2 * (log(Xb) + (1-Xb)/2)
    
    '''

    # Calculates the total density
    rho_tot = rho_total_array_AAAA + rho_total_array_BBBB + rho_total_array_AABB
    
    # To prevent error
    rho_A_floored = np.maximum(rho_total_array_AAAA, floor)
    rho_B_floored = np.maximum(rho_total_array_BBBB, floor)
    rho_AB_floored = np.maximum(rho_total_array_AABB, floor)
    rho_tot_floored = np.maximum(rho_tot, floor)

    # Finds f_ref including the final term
    f_ref = (rho_tot_floored * np.log(v0 * rho_tot_floored) - rho_tot_floored + B2 * rho_tot_floored**2 + 
             rho_A_floored  * np.log(rho_A_floored  / rho_tot_floored) + rho_B_floored  * np.log(rho_B_floored  / rho_tot_floored) + rho_AB_floored * np.log(rho_AB_floored / rho_tot_floored))
    
    # Finds f_b for each component and adds
    f_b = (rho_A_floored  * 4 * (np.log(Xa) + (1-Xa)/2) + rho_B_floored  * 4 * (np.log(Xb) + (1-Xb)/2) + 
           rho_AB_floored * 2 * (np.log(Xa) + (1-Xa)/2) +    rho_AB_floored * 2 * (np.log(Xb) + (1-Xb)/2))
    
    # Finds the free energy at each point (f{rho_i})
    f_local = f_ref + f_b
    
    # Calculates the gradient term K/2 * sum_i (gradient rho_i)^2
    f_gradient = 0.5 * K * ((np.gradient(rho_A_floored, dx))**2 + (np.gradient(rho_B_floored, dx))**2 + (np.gradient(rho_AB_floored, dx))**2)
    
    # Total free energy density
    f_total = f_local + f_gradient
    
    # Integrate over volume in 1D (multiply by dx and sum)
    F_total = np.trapz(f_total, dx=dx)
    
    return F_total

In [ ]:
# Initialises system with dilute and dense phases already
def coextanh(gridpoints,rho_dilute,rho_dense,interface_width=5):
    x_values=np.arange(gridpoints)
    return (rho_dilute-rho_dense)*(np.tanh((np.fabs(x_values-gridpoints*0.5)-gridpoints*0.25)/interface_width)+1)/2.+rho_dense

cx_values = list(np.linspace(0, 2, 21))


# Loads Spinodal Data
spinodal_data = np.loadtxt(f"spinodals/f{valence:.1f}")
idx_spinodal = np.argmin(np.abs(spinodal_data[:,0] - T))
rho_spinodal_low  = spinodal_data[idx_spinodal, 1]
rho_spinodal_high = spinodal_data[idx_spinodal, 2]

rho_spinodal_low  = spinodal_data[idx_spinodal, 1]
rho_spinodal_high = spinodal_data[idx_spinodal, 2]

rho_A_final_list = []
rho_B_final_list = []
rho_AB_final_list = []


for cx in cx_values:
    Cx = cx

        
    rhoAAAA = coextanh(gridpoints,rho_spinodal_low,rho_spinodal_high)
    rhoBBBB = np.roll(rhoAAAA, gridpoints//2)

    rho_AB_relative = rhoAAAA * rhoBBBB
    rhoAABB = (rho_AB_relative * cx * np.sum(rhoAAAA))/(np.sum(rho_AB_relative))

    if np.isclose(np.sum(rhoAABB)/np.sum(rhoAAAA), cx) == False:

        print(f"ERROR: CX IS ACTUALLY {np.sum(rhoAABB)/np.sum(rhoAAAA)} not {cx}" )


    # Initializes final rho arrays for graphing
    rho_total_array_AAAA = rhoAAAA 
    rho_total_array_BBBB = rhoBBBB
    rho_total_array_AABB = rhoAABB


    for step in range(duration):

        # Computes a step
        rhoAAAA_step, rhoBBBB_step, rhoAABB_step, Xa, Xb = compute_step_three(rhoAAAA, rhoBBBB, rhoAABB)
        rhoAAAA += rhoAAAA_step
        rhoBBBB += rhoBBBB_step
        rhoAABB += rhoAABB_step

        # Adds the new density to the array of densities + checks mass conservation every 10^5 steps
        if step % (save_interval) == 0:
            rho_total_array_AAAA = np.vstack((rho_total_array_AAAA, rhoAAAA))
            rho_total_array_BBBB = np.vstack((rho_total_array_BBBB, rhoBBBB))
            rho_total_array_AABB = np.vstack((rho_total_array_AABB, rhoAABB))

            
            print(f"Progress: {(step/(save_interval))} out of {duration/(save_interval)} for cx = {cx}")

    rho_A_final_list.append(rhoAAAA)
    rho_B_final_list.append(rhoBBBB)
    rho_AB_final_list.append(rhoAABB)





Progress: 0.0 out of 500.0 for cx = 0.0
Progress: 1.0 out of 500.0 for cx = 0.0
Progress: 2.0 out of 500.0 for cx = 0.0
Progress: 3.0 out of 500.0 for cx = 0.0
Progress: 4.0 out of 500.0 for cx = 0.0
Progress: 5.0 out of 500.0 for cx = 0.0
Progress: 6.0 out of 500.0 for cx = 0.0
Progress: 7.0 out of 500.0 for cx = 0.0
Progress: 8.0 out of 500.0 for cx = 0.0
Progress: 9.0 out of 500.0 for cx = 0.0
Progress: 10.0 out of 500.0 for cx = 0.0
Progress: 11.0 out of 500.0 for cx = 0.0
Progress: 12.0 out of 500.0 for cx = 0.0
Progress: 13.0 out of 500.0 for cx = 0.0
Progress: 14.0 out of 500.0 for cx = 0.0
Progress: 15.0 out of 500.0 for cx = 0.0
Progress: 16.0 out of 500.0 for cx = 0.0
Progress: 17.0 out of 500.0 for cx = 0.0
Progress: 18.0 out of 500.0 for cx = 0.0
Progress: 19.0 out of 500.0 for cx = 0.0
Progress: 20.0 out of 500.0 for cx = 0.0
Progress: 21.0 out of 500.0 for cx = 0.0
Progress: 22.0 out of 500.0 for cx = 0.0
Progress: 23.0 out of 500.0 for cx = 0.0
Progress: 24.0 out of 500.

In [ ]:
surface_tensions = []


for i, cx in enumerate(cx_values):
    Cx = cx

    rhoAAAA = np.array(rho_A_final_list[i])
    rhoBBBB = np.array(rho_B_final_list[i])
    rhoAABB = np.array(rho_AB_final_list[i])

    _, _, _, Xa, Xb = compute_step_three(rhoAAAA, rhoBBBB, rhoAABB)

    #Takes floored value for densities and finds area of interface for final system
    rho_A_floored = np.maximum(rhoAAAA, floor)
    rho_B_floored = np.maximum(rhoBBBB, floor)
    rho_AB_floored = np.maximum(rhoAABB, floor)
    rho_tot = rhoAAAA + rhoBBBB
    rho_tot_floored = np.maximum(rho_tot, floor)
    A = dx ** 2

    # Extracts dilute/dense phase densities for each component and creates homogeneous systems
    rho_A_dilute = np.min(rho_A_floored) 
    rho_A_dense = np.max(rho_A_floored)
    rho_B_dilute = np.min(rho_B_floored)
    rho_B_dense = np.max(rho_B_floored)
    rho_AB_dilute = np.min(rho_AB_floored)
    rho_AB_dense  = np.max(rho_AB_floored)

    rho_A_dilute_system = np.full(gridpoints, rho_A_dilute)
    rho_A_dense_system = np.full(gridpoints, rho_A_dense)
    rho_B_dilute_system = np.full(gridpoints, rho_B_dilute)
    rho_B_dense_system = np.full(gridpoints, rho_B_dense)
    rho_AB_dilute_system = np.full(gridpoints, rho_AB_dilute)
    rho_AB_dense_system  = np.full(gridpoints, rho_AB_dense)

    # Finds Xa and Xb for dilute, dense, and coex systems

    # For coexisting system
    Xa_final_coex = np.maximum(Xa, floor)
    Xb_final_coex = np.maximum(Xb, floor) 

    # For dilute system
    CaDa_dilute = 4*rho_A_dilute_system*Da 
    CaDa_dilute_floored = np.maximum(CaDa_dilute, floor)
    Xa_final_dilute = (-1 + np.sqrt(1 + 4 * CaDa_dilute_floored)) / (2 * CaDa_dilute_floored)

    CaDb_dilute = 4*rho_B_dilute_system*Db 
    CaDb_dilute_floored = np.maximum(CaDb_dilute, floor)
    Xb_final_dilute = (-1 + np.sqrt(1 + 4 * CaDb_dilute_floored)) / (2 * CaDb_dilute_floored)


    # For dilute system
    CaDa_dilute = 4*(rho_A_dilute_system + 0.5*rho_AB_dilute_system)*Da 
    CaDa_dilute_floored = np.maximum(CaDa_dilute, floor)
    Xa_final_dilute = (-1 + np.sqrt(1 + 4 * CaDa_dilute_floored)) / (2 * CaDa_dilute_floored)

    CaDb_dilute = 4*(rho_B_dilute_system + 0.5*rho_AB_dilute_system)*Db 
    CaDb_dilute_floored = np.maximum(CaDb_dilute, floor)
    Xb_final_dilute = (-1 + np.sqrt(1 + 4 * CaDb_dilute_floored)) / (2 * CaDb_dilute_floored)

    # For dense system
    CaDa_dense = 4*(rho_A_dense_system + 0.5*rho_AB_dense_system)*Da 
    CaDa_dense_floored = np.maximum(CaDa_dense, floor)
    Xa_final_dense = (-1 + np.sqrt(1 + 4 * CaDa_dense_floored)) / (2 * CaDa_dense_floored)

    CaDb_dense = 4*(rho_B_dense_system + 0.5*rho_AB_dense_system)*Db 
    CaDb_dense_floored = np.maximum(CaDb_dense, floor)
    Xb_final_dense = (-1 + np.sqrt(1 + 4 * CaDb_dense_floored)) / (2 * CaDb_dense_floored)

    # Finds free energy of dilute, dense, and coexisting systems
    F_dilute = calc_free_energy_multicomponent_AB(rho_A_dilute_system, rho_B_dilute_system, rho_AB_dilute_system,
                                            Xa_final_dilute, Xb_final_dilute)

    F_dense = calc_free_energy_multicomponent_AB(rho_A_dense_system, rho_B_dense_system, rho_AB_dense_system,
                                            Xa_final_dense, Xb_final_dense)

    F_coex = calc_free_energy_multicomponent_AB(rho_A_floored, rho_B_floored,  rho_AB_floored,
                                        Xa_final_coex, Xb_final_coex)

    # Finds number of interfaces, assuming a proper phase separated final rho using total density
    threshold = (np.min(rho_tot_floored) + np.max(rho_tot_floored)) / 2
    interface_locations = np.where(np.diff((rho_tot > threshold).astype(int)) != 0) #Assigns dilute/dense, then finds changes between them
    interface_count = len(interface_locations[0]) #total number of interfaces (should be 2)

    dilute_fraction = np.sum(rho_tot < threshold) / gridpoints
    dense_fraction = np.sum(rho_tot >= threshold) / gridpoints

    surface_tension = (F_coex - dilute_fraction * F_dilute - dense_fraction * F_dense) / (interface_count * A)    
    print(f"The surface tension is {surface_tension:.6e} for T = {T} in a 1D system with {interface_count} interfaces and 3 components (AAAA, BBBB, and AABB with Cx = {Cx}) initialized with a nonuniform distribution")

    surface_tensions.append(surface_tension)

In [ ]:
plt.plot(cx_values, surface_tensions)

plt.xlabel("Relative Crosslinker Concentration")
plt.ylabel(r"Surface Tension (${k_B T}/{n m ^2}$)")
plt.yscale("log")

plt.axhline(0.0015)

In [ ]:
# Initializes x-axis values
x = np.arange(rho_total_array_AAAA.shape[1])

# Picks evenly spaced timesteps to graph
n_snapshots = len(cx_values)

# Makes subplots
fig, axes = plt.subplots(n_snapshots, 1, figsize=(10, 2*n_snapshots), sharex=True)

colors = {"AAAA": "firebrick", "BBBB": "mediumseagreen", "AABB": "darkblue"}

for i in range(n_snapshots):
    ax = axes[i]
    ax.plot(x, rho_A_final_list[i], color=colors["AAAA"], label="AAAA", alpha=0.5)
    ax.plot(x, rho_B_final_list[i], color=colors["BBBB"], label="BBBB", alpha=0.5)
    ax.plot(x, rho_AB_final_list[i], color=colors["AABB"], label="AABB", alpha=0.5)

    ax.set_ylabel("rho (nm⁻³)")
    ax.set_title(f"cx = {cx_values[i]}")
    ax.legend(loc="upper right", fontsize=8)

fig.suptitle("Multiple Component System (A, B, AB), Uniform Initial Distribution")
axes[-1].set_xlabel("Position x")
plt.tight_layout()
plt.show()